# Paper 10 backbone on YOLO26n: C2f_GhostECA

| Part | Source |
|---|---|
| **Backbone (layers 0–10)** | **Paper 10:** `C3k2` at layers 2, 4, 6, 8 replaced by **C2f_GhostECA**; SPPF and C2PSA are YOLO26n base |
| Neck FPN + PAN (layers 11–22) | YOLO26n base, unchanged |
| Head (Detect, layer 23) | YOLO26n base, unchanged (Paper 10 keeps the native YOLO26 head) |

The baseline ("Standard YOLO26n Backbone") is **not trained here**; its values are already filled in the final table.

**Kaggle settings:** Accelerator = **GPU T4 x1**, Internet = **On**, keep the notebook **Private**

## 1. Setup

In [ ]:
# ultralytics version is pinned: use the same version on Kaggle, Raspberry Pi 5 and Jetson
!pip install -q ultralytics==8.4.137 roboflow

In [ ]:
import os, sys, gc, shutil, random
from copy import deepcopy
from pathlib import Path

import numpy as np
import pandas as pd
import torch, torchvision
import ultralytics

print("ultralytics", ultralytics.__version__, "| torch", torch.__version__, "| torchvision", torchvision.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE (turn on the GPU accelerator)")

WORK = Path("/kaggle/working")
os.chdir(WORK)
sys.path.insert(0, str(WORK))  # so ghosteca_backbone.py can be imported

## 2. Dataset (download from Roboflow)

The Roboflow API key is written in the cell below. Keep this notebook **Private** on Kaggle.

In [ ]:
import os, yaml
from roboflow import Roboflow

os.chdir("/kaggle/working")

# Roboflow API key (keep this notebook Private on Kaggle)
API_KEY = "CxHgc1Nn3FK5vYNsBtJW"

rf = Roboflow(api_key=API_KEY)
project = rf.workspace("md-mahibullah").project("digitalaccess-2-copy")
dataset = project.version(4).download("yolo26")  # Roboflow YOLO26 export (same label format as YOLOv8)

# Rename and fix paths
new = "/kaggle/working/digitalaccess"
if os.path.exists(dataset.location) and not os.path.exists(new):
    os.rename(dataset.location, new)

# Update YAML
yaml_path = f"{new}/data.yaml"
with open(yaml_path) as f:
    data = yaml.safe_load(f)
data["path"], data["train"], data["val"] = new, "train/images", "valid/images"
if os.path.exists(f"{new}/test/images"):
    data["test"] = "test/images"
else:
    data.pop("test", None)
with open(yaml_path, "w") as f:
    yaml.dump(data, f)

# the following cells use these variables
DATA_ROOT = Path(new)
DATA_YAML = yaml_path
names = data["names"]
assert data.get("test"), "No test split: the evaluation and FPS steps need a test split"

for s in ["train", "valid", "test"]:
    print(f"{s:6s}: {len(list((DATA_ROOT / s / 'images').glob('*')))} images")
print("nc =", len(names))
print("✅ Dataset prepared!")

## 3. Paper 10 backbone: C2f_GhostECA

**Building blocks** (Paper 10, Sections 2.2–2.4).
- **EECA (Enhanced ECA):** global average pooling → 1-D conv across channels with kernel **7** → sigmoid → multiply with the input.
  No channel reduction; only 7 weights.
- **GhostBottleneck_ECA:** GhostConv 3×3 → GhostConv 3×3 → EECA → residual add.
  Each GhostConv makes half of its output with a normal conv and the other half with a cheap depthwise conv.
- **C2f_GhostECA:** C2f structure (1×1 conv, split, bottlenecks, concat of all parts, 1×1 conv) with GhostBottleneck_ECA inside.

**Where it sits in YOLO26n.**

| Layer | Base YOLO26n | Paper 10 backbone | Output |
|---|---|---|---|
| 2 | C3k2 | **C2f_GhostECA** | P2/4 (160 × 160) |
| 4 | C3k2 | **C2f_GhostECA** | P3/8 (80 × 80) |
| 6 | C3k2 (c3k) | **C2f_GhostECA** | P4/16 (40 × 40) |
| 8 | C3k2 (c3k) | **C2f_GhostECA** | P5/32 (20 × 20) |

Hidden width and number of blocks are the same as the base C3k2, so input and output channels stay the same and the
neck connections do not change.

**Pretrained weights.** `cv1` and `cv2` of C2f_GhostECA have exactly the same shapes as in the base C3k2, so both are
copied from `yolo26n.pt`. Only the GhostBottleneck_ECA blocks start from scratch.

**Not taken from Paper 10:** SlimSPPF (by the paper's own equations it is the same as the YOLO26 SPPF) and the
re-scaled depth/width (the values are not given).

**NCNN note.** The 1-D channel conv of EECA is written as a Conv2d with kernel (7, 1) on a 4-D tensor.

**Assumptions.** Paper 10 does not give the GhostConv kernel sizes; here the primary conv is 3×3 and the cheap
depthwise conv is 5×5 (ultralytics `GhostConv` default). Copying the pretrained `cv1`/`cv2` is added in this notebook.

In [ ]:
%%writefile ghosteca_backbone.py
"""Paper 10 backbone for YOLO26n: C3k2 at layers 2, 4, 6, 8 -> C2f_GhostECA (GhostBottleneck + Enhanced ECA, k=7)."""
import torch
import torch.nn as nn

from ultralytics.models.yolo.detect import DetectionTrainer
from ultralytics.nn.modules.block import C3k2
from ultralytics.nn.modules.conv import Conv, GhostConv
from ultralytics.nn.tasks import DetectionModel
from ultralytics.utils import RANK

GHOSTECA_LAYERS = (2, 4, 6, 8)


class EECA(nn.Module):
    """Enhanced ECA: global avg pool -> conv across channels (k=7) -> sigmoid -> scale. 1-D conv written as 4-D Conv2d."""

    def __init__(self, k=7):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.conv = nn.Conv2d(1, 1, (k, 1), padding=(k // 2, 0), bias=False)
        self.act = nn.Sigmoid()

    def forward(self, x):
        y = self.pool(x).transpose(1, 2)  # (B, C, 1, 1) -> (B, 1, C, 1)
        y = self.act(self.conv(y)).transpose(1, 2)  # back to (B, C, 1, 1)
        return x * y


class GhostBottleneckECA(nn.Module):
    """GhostConv 3x3 -> GhostConv 3x3 -> EECA, with a residual connection."""

    def __init__(self, c, k=3):
        super().__init__()
        self.cv1 = GhostConv(c, c, k)
        self.cv2 = GhostConv(c, c, k)
        self.eca = EECA(7)

    def forward(self, x):
        return x + self.eca(self.cv2(self.cv1(x)))


class C2fGhostECA(nn.Module):
    """C2f with GhostBottleneck_ECA blocks: cv1 -> split -> n blocks -> concat all -> cv2."""

    def __init__(self, c1, c2, n=1, c=None):
        super().__init__()
        self.c = c or c2 // 2  # hidden channels
        self.cv1 = Conv(c1, 2 * self.c, 1, 1)
        self.cv2 = Conv((2 + n) * self.c, c2, 1)
        self.m = nn.ModuleList(GhostBottleneckECA(self.c) for _ in range(n))

    def forward(self, x):
        y = list(self.cv1(x).chunk(2, 1))
        y.extend(m(y[-1]) for m in self.m)
        return self.cv2(torch.cat(y, 1))


def add_ghosteca(model, layers=GHOSTECA_LAYERS):
    """Replace C3k2 at the given layers with C2f_GhostECA, copying cv1 and cv2 (same shapes). Returns the model."""
    for i in layers:
        old = model.model[i]
        assert isinstance(old, C3k2), f"layer {i} is {type(old).__name__}, not C3k2"
        new = C2fGhostECA(old.cv1.conv.in_channels, old.cv2.conv.out_channels, n=len(old.m), c=old.c)
        new.cv1.load_state_dict(old.cv1.state_dict())
        new.cv2.load_state_dict(old.cv2.state_dict())
        new = new.to(next(old.parameters()).device)
        new.i, new.f, new.type = old.i, old.f, "C2fGhostECA"  # keep the routing attributes used by the model's forward
        new.np = sum(p.numel() for p in new.parameters())
        model.model[i] = new
    return model


class GhostECATrainer(DetectionTrainer):
    """Loads the pretrained weights first, then swaps C3k2 -> C2f_GhostECA (cv1/cv2 keep the pretrained weights)."""

    def get_model(self, cfg=None, weights=None, verbose=True):
        model = DetectionModel(cfg, nc=self.data["nc"], ch=self.data["channels"], verbose=False)
        if weights:
            model.load(weights)
        add_ghosteca(model)
        if RANK in {-1, 0}:
            print(f"Paper 10 backbone: C2f_GhostECA at layers {GHOSTECA_LAYERS}")
        if verbose and RANK in {-1, 0}:
            model.info()
        return model


In [ ]:
import importlib, ghosteca_backbone
importlib.reload(ghosteca_backbone)
from ghosteca_backbone import add_ghosteca, GhostECATrainer
from ultralytics.nn.tasks import DetectionModel, yaml_model_load
from ultralytics.utils.torch_utils import get_flops

def fused_params_gflops(model, imgsz=640):
    m = deepcopy(model).float().cpu().eval()
    m.fuse(verbose=False)
    return sum(p.numel() for p in m.parameters()) / 1e6, get_flops(m, imgsz)

# structure check (untrained): base vs Paper 10 backbone
base = DetectionModel(yaml_model_load("yolo26n.yaml"), nc=len(names), verbose=False).eval()
ghost = add_ghosteca(deepcopy(base)).eval()
with torch.no_grad():
    _ = ghost(torch.zeros(1, 3, 640, 640))  # forward check
print("layers 2, 4, 6, 8:", [type(ghost.model[i]).__name__ for i in (2, 4, 6, 8)])
print("hidden width c   :", [ghost.model[i].c for i in (2, 4, 6, 8)])
for tag, m in (("base", base), ("Paper 10 backbone", ghost)):
    pm, gf = fused_params_gflops(m)
    print(f"{tag:18s} Params {pm:.3f} M | GFLOPs {gf:.2f}")

## 4. Training

Only the Paper 10 backbone model is trained.

In [ ]:
from ultralytics import YOLO

EPOCHS  = 100
IMGSZ   = 640
BATCH   = 16
WORKERS = 2
SEED    = 0
PROJECT = str(WORK / "runs")

COMMON = dict(
    data=DATA_YAML, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, seed=SEED, deterministic=True,
    pretrained="yolo26n.pt", device=0, workers=WORKERS, cache=False, patience=30, cos_lr=False,
    project=PROJECT, exist_ok=True, plots=True,
    # channels_last=False: ultralytics' MuSGD crashes on channels_last weights ("view size is not compatible").
    optimizer="MuSGD", lr0=0.01, momentum=0.9, channels_last=False,
)

# (row label in the table, run name)
VARIANTS = [
    ("C2f_GhostECA Backbone (Paper 10)", "bb_ghosteca"),
]

def best_path(run):
    return Path(PROJECT) / f"{run}_s{SEED}" / "weights" / "best.pt"

for label, run in VARIANTS:
    print(f"\n========== {label} ({run}) ==========")
    model = YOLO("yolo26n.yaml")
    model.train(trainer=GhostECATrainer, **dict(COMMON, name=f"{run}_s{SEED}"))
    del model; gc.collect(); torch.cuda.empty_cache()

## 5. Test-set evaluation: Precision, Recall, F1-score, mAP@50, Params, GFLOPs

Params and GFLOPs are counted on the fused inference model, the same way as the baseline row.

In [ ]:
import ghosteca_backbone  # must be imported before loading a C2f_GhostECA checkpoint

rows = []
for label, run in VARIANTS:
    w = best_path(run)
    if not w.exists():
        print(f"{run}: best.pt not found, train it first"); continue
    r = YOLO(str(w)).val(data=DATA_YAML, split="test", imgsz=IMGSZ, batch=16, device=0, plots=False, verbose=False)
    P, R = float(r.box.mp), float(r.box.mr)
    params, gflops = fused_params_gflops(YOLO(str(w)).model, IMGSZ)
    rows.append({"run": run, "Backbone Variant": label,
                 "Params (M)": round(params, 2), "GFLOPs": round(gflops, 2),
                 "Precision": round(P, 3), "Recall": round(R, 3),
                 "F1-score": round(2 * P * R / (P + R + 1e-9), 3),
                 "mAP@50": round(float(r.box.map50), 3)})
METRICS = pd.DataFrame(rows)
METRICS.drop(columns="run")

## 6. Package for Raspberry Pi 5 and Jetson

Creates `paper10_backbone_edge_bench.zip` in the **Output** tab with:

| Item | Used on |
|---|---|
| `bb_ghosteca_ncnn_model/` (NCNN model, imgsz 640) | Raspberry Pi 5 |
| `bb_ghosteca.pt` | Jetson |
| `bench_images/` (50 real test images, seed 0) | both |
| `ghosteca_backbone.py` | Jetson (needed to load the C2f_GhostECA `.pt`) |
| `pi5_bench.py`, `jetson_bench.py` | benchmark scripts |

Both scripts: batch 1, imgsz 640, cycle through the 50 real test images (loaded into memory first),
100 warm-up runs, then 1000 timed runs end-to-end (preprocessing + inference + postprocessing), and print the **median** ms and FPS.
Unzip into its own folder, e.g. `python -m zipfile -e paper10_backbone_edge_bench.zip paper10_backbone_edge_bench`.

In [ ]:
EDGE_IMGSZ = 640
N_BENCH_IMAGES = 50
EDGE = WORK / "paper10_backbone_edge_bench"
shutil.rmtree(EDGE, ignore_errors=True)
EDGE.mkdir()
shutil.copy("ghosteca_backbone.py", EDGE / "ghosteca_backbone.py")

# 50 real test images (fixed seed, so every model and device uses the same images)
test_imgs = sorted(p for p in (DATA_ROOT / "test" / "images").glob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"})
random.Random(SEED).shuffle(test_imgs)
(EDGE / "bench_images").mkdir()
for p in test_imgs[:N_BENCH_IMAGES]:
    shutil.copy(p, EDGE / "bench_images" / p.name)
print(f"bench_images: {len(list((EDGE / 'bench_images').glob('*')))} real test images")

for _, run in VARIANTS:
    w = best_path(run)
    if not w.exists():
        print(f"{run}: best.pt not found, skipped"); continue
    shutil.copy(w, EDGE / f"{run}.pt")
    p = YOLO(str(w)).export(format="ncnn", imgsz=EDGE_IMGSZ, half=False)
    shutil.copytree(p, EDGE / f"{run}_ncnn_model")  # ultralytics needs the "_ncnn_model" suffix
    print(f"{run:12s} -> {run}.pt, {run}_ncnn_model/")

COMMON_BENCH = """
import glob, statistics, sys, time
import cv2

IMGSZ, WARMUP, RUNS = {imgsz}, 100, 1000
imgs = [cv2.imread(p) for p in sorted(glob.glob("bench_images/*"))]
imgs = [im for im in imgs if im is not None]
if not imgs:
    sys.exit("No images found in bench_images/. Run this script inside the unzipped folder.")
print(f"{{len(imgs)}} real test images, imgsz {{IMGSZ}}, batch 1, {{WARMUP}} warm-up + {{RUNS}} timed runs (median)")


def bench(model, **kw):
    for i in range(WARMUP):
        model.predict(imgs[i % len(imgs)], imgsz=IMGSZ, verbose=False, **kw)
    times = []
    for i in range(RUNS):
        t = time.perf_counter()
        model.predict(imgs[i % len(imgs)], imgsz=IMGSZ, verbose=False, **kw)
        times.append((time.perf_counter() - t) * 1000)
    return statistics.median(times)
""".format(imgsz=EDGE_IMGSZ)

PI5 = '''# Raspberry Pi 5 benchmark (NCNN). Run inside the unzipped folder:  python pi5_bench.py
from ultralytics import YOLO
''' + COMMON_BENCH + '''
models = sorted(glob.glob("*_ncnn_model"))
if not models:
    sys.exit("No *_ncnn_model folders found.")
for path in models:
    ms = bench(YOLO(path, task="detect"))
    print(f"{path.replace('_ncnn_model', ''):12s}  median {ms:8.2f} ms   {1000 / ms:7.2f} FPS")
'''

JETSON = '''# Jetson benchmark (PyTorch, CUDA FP16). Run inside the unzipped folder:  python jetson_bench.py
import sys
import torch

if not torch.cuda.is_available():
    sys.exit("WARNING: CUDA GPU not available. Install the NVIDIA Jetson PyTorch wheel for your JetPack, then run again.")
print("GPU:", torch.cuda.get_device_name(0))

sys.path.insert(0, ".")
import ghosteca_backbone  # needed to load the C2f_GhostECA checkpoint
from ultralytics import YOLO
''' + COMMON_BENCH + '''
models = sorted(glob.glob("*.pt"))
if not models:
    sys.exit("No .pt files found.")
for path in models:
    ms = bench(YOLO(path, task="detect"), device=0, half=True)
    print(f"{path.replace('.pt', ''):12s}  median {ms:8.2f} ms   {1000 / ms:7.2f} FPS")
'''

(EDGE / "pi5_bench.py").write_text(PI5)
(EDGE / "jetson_bench.py").write_text(JETSON)
shutil.make_archive(str(WORK / "paper10_backbone_edge_bench"), "zip", EDGE)
print("\nDownload paper10_backbone_edge_bench.zip from the Output tab.")

### How to run on the devices

**Raspberry Pi 5** (same Pi and same conditions as the baseline measurement)
1. Activate the Python environment that has `ultralytics==8.4.137` and `ncnn`.
2. `echo performance | sudo tee /sys/devices/system/cpu/cpu*/scaling_governor`
3. Go into the unzipped folder and run `python pi5_bench.py`.
4. Run it 3 times with a 2–3 minute pause; use the middle value. Check `vcgencmd measure_temp` and `vcgencmd get_throttled` (`throttled=0x0` = OK).

**Jetson**
1. NVIDIA Jetson PyTorch + torchvision for your JetPack, then `ultralytics==8.4.137`.
2. `sudo nvpmodel -m 0` (check with `sudo nvpmodel -q`) and `sudo jetson_clocks`.
3. Go into the unzipped folder and run `python jetson_bench.py`. Run 3 times; use the middle value.

## 7. Final backbone table

The baseline row is already filled in. Type the median FPS of the Paper 10 backbone model
(and the baseline Jetson FPS when you have it), keep `None` for a missing value (shown as `--`), then run the cell.
Run section 5 first in the same session (it fills `METRICS`).

In [ ]:
# baseline values (identical YOLO26n model)
BASE_ROW = {"Backbone Variant": "Standard YOLO26n Backbone", "Params (M)": 2.38, "GFLOPs": 5.34,
            "Precision": 0.971, "Recall": 0.937, "F1-score": 0.953, "mAP@50": 0.958,
            "FPS (Rasp-5)": 13.38, "FPS (Jetson)": None}

FPS_PI5    = {"bb_ghosteca": None}
FPS_JETSON = {"bb_ghosteca": None}

table = METRICS.copy()
table["FPS (Rasp-5)"] = table["run"].map(lambda r: FPS_PI5.get(r))
table["FPS (Jetson)"] = table["run"].map(lambda r: FPS_JETSON.get(r))
COLS = ["Backbone Variant", "Params (M)", "GFLOPs", "Precision", "Recall", "F1-score", "mAP@50", "FPS (Rasp-5)", "FPS (Jetson)"]
table = pd.concat([pd.DataFrame([BASE_ROW]), table[COLS]], ignore_index=True)[COLS]
table = table.astype(object).where(table.notna(), "--")
table.style.hide(axis="index")

## 8. Zip all results

Run this last. It creates **`paper10_backbone_all_results.zip`** in the **Output** tab with:

| Folder in the zip | Contents |
|---|---|
| `paper10_backbone_edge_bench/` | NCNN + `.pt` model, 50 bench images, `ghosteca_backbone.py`, `pi5_bench.py`, `jetson_bench.py` |
| `weights/` | `best.pt`, renamed `bb_ghosteca_best.pt` |
| `runs/` | full training folder: `best.pt`, `last.pt`, `results.csv`, training curves, confusion matrix, PR curves |

In [ ]:
import zipfile

ALL_ZIP = WORK / "paper10_backbone_all_results.zip"
folders = [WORK / "paper10_backbone_edge_bench", WORK / "runs"]

with zipfile.ZipFile(ALL_ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for folder in folders:
        if not folder.exists():
            print(f"{folder.name}: not found, skipped"); continue
        n = 0
        for f in folder.rglob("*"):
            if f.is_file():
                z.write(f, f.relative_to(WORK))
                n += 1
        print(f"{folder.name + '/':30s} {n} files")
    for _, run in VARIANTS:
        w = best_path(run)
        if w.exists():
            z.write(w, f"weights/{run}_best.pt")
            print(f"weights/{run}_best.pt")

print(f"\nDownload {ALL_ZIP.name} ({ALL_ZIP.stat().st_size / 1e6:.1f} MB) from the Output tab.")